# Energy budget — T4, the four numbers per circuit

H_max (the per-event cap from circuits.yaml, E-02), E_dep from T1, and the two harvest
mechanisms — braking recovery and 250 kW super-clipping on throttle (AR-07) — are the
entire energy game (F-02/F-04). Supply is what the mechanisms deliver capped by H_max;
the shortfall against E_dep is the derate deficit that ranks circuits by straight-line
energy stress. Lift-and-coast (AR-06) buys extra harvest by spending lap time; c_lc is
the fitted price, and the cross-circuit median fills `conserve_pace_cost_s_per_mj` in
physics_2026.yaml. Logic lives in `senna.physics.synth.energy_budget`.

In [ ]:
import argparse
import asyncio
import os

from senna import config

synth_cfg = config.load("synth")
physics_2026 = config.load("physics_2026")
CALIB = config.get(config.load("data_split"), "calibration_rounds")

# the battery needs Postgres + the FastF1 cache; export DATABASE_URL first
assert os.environ.get("DATABASE_URL"), "export DATABASE_URL (see .env.example) before running"

In [ ]:
from senna.physics.synth import run

args = argparse.Namespace(only="aero,pointmass,energy", circuit="monaco_grand_prix",
                          transforms_out=None, physics_2026_out=None)
transforms = asyncio.run(run.run_battery(args))
transforms["circuits"]["monaco_grand_prix"]["t4_energy"]

In [ ]:
# c_lc is derived from the calibrated point-mass profile — a lift-and-coast sweep,
# not a regression on noisy race laps — so it is deterministic and defensible (T4)
config.get(physics_2026, "harvest.lift_and_coast_kw"), config.get(physics_2026, "harvest.super_clipping_kw")